<center><b>Python Programming for Multilingual Texts</b></center>
<center>1-1: From Primary Source to Data</center>

---

# Introduction to The *Montagu Corpus*

## Origin of the Primary Source

This Jupyter Notebook is a step-by-step tutorial of the text extraction and dataset building process for the primary source: *The Letters of Lady Montagu*, originally published in 1763. Throughout this workshop, we used the 1790 edition of the same text, edited by Desmond Grocott and published by the **Project Gutenberg** [ebook ID: 17520](https://www.gutenberg.org/ebooks/17520) on January 15, 2006.

> You can find the 1790 edition in this repository, both in its [original PDF](../../readings/primary_sources/montagu_1790.pdf) format and in Project Gutenberg editions [as a text file](../../data/pg17520.txt) and [as an ePub](../../data/pg17520.epub).

<figure style="text-align: center; margin: 1.5em auto;">
  <img src="../../img/montagu_letters_1790_cover.png" width="550" alt="Cover of the 1790 edition of Montagu's Letters">
  <figcaption style="font-style: italic; opacity: 0.6; margin-top: 0.5em;">
    Title page of the 1790 edition of <em>Letters of Lady Mary Wortley Montagu</em>, published in London.
  </figcaption>
</figure>

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
  From Early Modern print to digital — the 1790 printed edition (above) and its Project Gutenberg digitization (below).
</div>

<figure style="text-align: center; margin: 1.5em auto;">
  <img src="../../img/montagu_1790_gutenberg_page.png" width="550" alt="Project Gutenberg page for eBook #17520">
  <figcaption style="font-style: italic; opacity: 0.6; margin-top: 0.5em;">
    The Project Gutenberg page for <em>Letters of Lady Mary Wortley Montagu</em>.
  </figcaption>
</figure>

## Overview

This notebook consists of three sections:

1. **Text Extraction** — How we identified the relevant sections from the Gutenberg edition and split one continuous text into individual letter files and a CSV where each row is one letter.

2. **Editing** — The minimum post-processing required to prepare the corpus for computational text analysis.

3. **Data Enrichment** — Optional yet highly important interventions that improve the analytical potential of this corpus as a dataset.


## Text Extraction

### Step 0: OCR?

Optical Character Recogtion (OCR) is the process through which images of text are turned into machine readable texts.

In [ ]:
import os
import matplotlib.pyplot as plt
from pypdf import PdfReader
from PIL import Image

Any questions about the libraries?

* [pypdf](https://pypi.org/project/pypdf/) A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
* [matplotlib](https://matplotlib.org/) a library for plotting data, here to display images of PDF pages and texts more neatly
* [PIL](https://pypi.org/project/pillow/) Python Imaging Library

<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;">Let's load our 1790 PDF to show how OCR works!</mark>


In [ ]:
# PDF file path to the Montagu 1790 edition
pdf_path = '../../readings/primary_sources/montagu_1790.pdf'

# Let's read it in
reader = PdfReader(pdf_path)

# To verify that it loaded: print number of pages
# I am expecting 241 pages
print(f"PDF loaded successfully. Total pages: {len(reader.pages)}")

In [ ]:
# We will now bring in the image of page 11

page_11 = Image.open("../../img/montagu_1790_page_11.png")
plt.figure(figsize=(10, 14))
plt.imshow(page_11)
plt.axis("off")
plt.show()

Now, this PDF downloaded from Google Books, comes with an embedded OCR layer. An educated guess is that during the scanning process, the high-tech scanners also ran a preliminary OCR model on these materials. Most standard OCR models are trained on contemporary texts with more standardized layout and spelling. They underperform heavily with historical texts.

> Let's take a closer look at Page 11 to discuss why OCR might not perform well with this work

<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;">Now, let's see what the embedded OCR layer actually contains</mark>

In [ ]:
# Bring in the embedded OCR
# Note that this is referring back pypdf from the first cell in this subsection 
# pypdf is 0 indexed, so we need to make sure that we read the correct page
ocr_text = reader.pages[21].extract_text()

# Visualize them side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Left: PDF image
axes[0].imshow(page_11)
axes[0].set_title('Page 11 - PDF Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Right: OCR text
axes[1].text(0.05, 0.95, ocr_text, transform=axes[1].transAxes, 
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             wrap=True)
axes[1].set_title('Embedded OCR Text Layer', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

Clearly, not great..

In [ ]:
# let's look at the beginning of the letter
ocr_text_lines = ocr_text.split('\n')
print(ocr_text_lines[1])

`makefuchſhortſtageseveryday,thatI ratherfancymyſelfuponpartiesofpleaſure`

<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;">OR</mark>


`make such short stages every day, that I rather fancy myself upon parties of pleasure`

We can certainly experiment with other OCR software to improve the quality of the text output. Or we can use the Project Gutenberg version of this text, which has been edited by a human.

> Please note that an human-edited clean text file might not be available for every text or every language. We are lucky that Lady Montagu's letters were popular enough to merit hundereds of hours of labor and resources. Same cannot be said for the large majority of texts created in non-English and under-resourced languages. 

### Step 1: Machine-Readable Text File

In [ ]:
# Download the txt file from Gutenberg and read it in

with open('../../data/pg17520.txt',"r", encoding="utf-8") as f:
    montagu_17520 = f.read()

print(montagu_17520)

### Step 2: Preprocessing

Historical data is not always very clean, even if it is already processed by contemporary scholars. In this example, we know that **we only want to extract the letters** but this edition contains copyright information from Project Gutenberg and introductory materials from the primary source itself. We prefer to remove these as a first step and often it is easier to do it manually.

This is where versioning comes into play. We do not want to lose our primary source, so we need to create copies.

1. Create a copy of `pg17520.txt` and name it `montagu_2006.txt`
2. Manually delete the copyright information and other details at the end of the text file added by Project Gutenberg
3. Manually remove the title page, preface, the advertisement by the editor, the verses, and the inquiry and save them to separate text files in our [data folder](../../data)


In [ ]:
# let's print the first 5000 characters
print(montagu_17520[:5000])

In [ ]:
# let's print from the 18600th character until the end
print(montagu_17520[-18600:])

<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;">We have some information from the Project Gutenberg both before and after the main text. We should come up with a plan to handle this.</mark>

> For project with larger datasets, such preliminary contents can be negligible but in our case, since we are very narrowly focusing on these letter, we need to do some preprocessing.


<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em 0; font-weight: 500;">
 Let's investigate what else is in this work other than the letters.
</div>

Before the first letter, there is a `title page`, a `preface` and an `advertisement`.

After the last letter, there are some additional materials and more importantly a `summary of the contents`. This is very typical of the era, a lot of early modern printed works had a summary of contents at the very end instead of the beginning of a book

In [ ]:
# this code identifies the character index of where the summary begins
# and prints the text starting in that index until 18200th character 

start_summary = montagu_17520.find("A SUMMARY OF THE CONTENTS.")
print(montagu_17520[start_summary:-18200])

The contents revealed that there are `58 letters` and `three additional texts`.

```md
1. Inquiry into the truth of Monsieur Rochefoucault's maxim, "That
marriage is sometimes convenient, but never delightful."

2. Verses written in the Chiask at Pera, overlooking Constantinople,
December 26th, 1718. By Lady Mary Wortley Montague.

3. Verses to Lady Mary Wortley Montague.  By Mr Pope.
```

In [ ]:
# Let's see them in detail
start_nonletter = montagu_17520.find("CONCERNING")
print(montagu_17520[start_nonletter:start_nonletter+20000])

These are three additional non-letter texts published with this work, which we will not incorporate into this corpus. Instead, we will save them as individual text files, like we did with the preface and the advertisement, to see if we can play around with them later down the line.

In [ ]:
montagu = open('../../data/montagu_2006.txt',"r", encoding="utf-8").read()

print(montagu[:5000])

### Step 3: Splitting the Source into Individual Letters

Now that we have singled out the text chunk with our letters, we can look into how we will be splitting this one continuious text into the 58 letters that it is internally divided into. This is not a simple task because the computer does not 'read' the way we do. It does not 'know' where a letter begins and another one ends. In other words:

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em auto; font-weight: 500; width: 90%;">
  How can we tell the computer that LETTER XXIII is not a random string of characters but actually a meaningful sub-unit of text?
</div>


The answer is `Regular Expressions` or `regex`

Later in this course, we will study regular expressions in more details. For the time being, let's keep things very simple.

<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;"> Regular Expressions (regex) are sequences of characters that define a search pattern. They are commonly used for pattern matching within strings, such as finding words, validating input, or replacing text.</mark>

In [ ]:
import re

In [ ]:
# let's read the first 1000 characters for a demonstration

montagu_1000 = montagu[:1000]
print(montagu_1000)

In [ ]:
# We can all mentions of years, expressed as 4 digits side-by-side

matches = re.findall(r"\d{4}", montagu_1000)
print("Years mentioned in the text:", matches)

In [ ]:
# We can find all instances of first person plural pronoun, we
matches = re.findall(r"we", montagu_1000)
print("We pronouns in text:", matches)

In [ ]:
# Hmm, we are missing some mentiones right?

import re

matches = re.findall(r"we", montagu_1000, flags=re.IGNORECASE)
print("We pronouns in text:", matches)

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em auto; font-weight: 500; width: 70%;">
  Now, in order to split the text by letter, we must find where a given letter begins.
</div>

In [ ]:
# We can express this in regex as LETTER followed by an Arabic or Roman numeral

matches = re.findall(r"LETTER [1-9IVXL]+", montagu_1000)
print("Letter headers found:", matches)

Seems promising, let's see if it scales

In [ ]:
matches = re.findall(r"LETTER [1-9IVXL]+", montagu)
print("Number of letter headers, expecting 58:", len(matches))
print("Letter headers found:", matches)

Well, it did not..

This makes a lot of sense, when we look at our text, we see that LETTER is more often than not shorted to LET or LET. We need a more inclusive regex

In [ ]:
matches = re.findall(r"LET(?:TER)?[.\s]+[IVXLCDM\d]+", montagu)
print("Number of letter headers, expecting 58:", len(matches))
print("Letter headers found:", matches)

<div style="border-left: 4px solid #60a5fa; padding: 0.5em 1em; margin: 1.5em 0;">
<strong>Note on Letter 11</strong>

<p>Letter 11 turned out to be more stubborn than necessary. The Gutenberg editor added spaces between the letters <code>L E T</code>. Upon closer investigation of the primary source, we decided that there was no particular reason for the spacing, so we just edited it out to make splitting easier.</p>

<img src="../../img/letter_11_text_edit.png" width="350" alt="Letter 11 text edit">
</div>


In [ ]:
## Let's scale this up and create our corpus

def split_letters(file_path, output_folder):
    # Ensure output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Read the entire text file
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # Use regex to find all letters (assuming they start with "LETTER. X")
    # Improved regex to handle "LETTER VIII", "LET. VIII", and other variations
    letters = re.split(r"(?=\nLET(?:TER)?(?:\.|\s)\s*[IVXLCDM\d]+)", text)

    # Remove any empty splits
    letters = [letter.strip() for letter in letters if letter.strip()]

    # Save each letter as a separate file
    for i, letter in enumerate(letters, start=1):
        file_name = f"letter_{i}.txt"
        with open(os.path.join(output_folder, file_name), "w", encoding="utf-8") as letter_file:
            letter_file.write(letter)

    print(f"Split {len(letters)} letters into individual files in '{output_folder}'.")

input_file = "../../data/montagu_2006.txt"  # Path to uploaded file
output_directory = "../../output/montagu_letters"  # Folder for individual letter files
split_letters(input_file, output_directory)

### Step 4: Parsing the letter structure 

In [ ]:
# Let's read one letter in and study its structure

with open("../../output/montagu_letters/letter_5.txt") as f:
    letter_5 = f.read()

print(letter_5)

Looking at Letter 5, we see the following:

1. We know (from our letter regex) that the all the extracted letters begin with the letter title (`LET. V.`). So, we can savely assume that the first line of the text is the letter title

2. We know that the addressee starts with a version of "to" (`TO THE COUNTESS OF B----.`). We created a regex that searches for versions of "to".

3. We know that the location and date information are in this format: "location, date" (`_Nuremberg, Aug_. 22. O. S. 1716.`). We also know that location, date is often the first line of text after the addressee line. We used this information to simplify our search in finding and splitting the location, date information.
    - At the same time we know that some letters have an editor's note between the addressee and the location,date column in this format: [Footnote: ....] Thus, we introduced a skipping condition to skip the line that contains 'Footnote:'

4. We designated everything after location, date section to be the body of the letter (`AFTER five days travelling post, ....`).


<mark style="background: rgba(167, 139, 250, 0.3); color: inherit; border-radius: 3px; padding: 0 2px;">Based on these assumptions, we will now create a CSV from this letter corpus, where each column contains a different metadata item: filename, title, addressee, location, date, and body and each row is a letter.</mark>


In [ ]:
# Let's create our CSV 
"""
Our goal is to create a CSV with the following columns:
filename, title, addressee, location, date, body

For example
letter_5.txt, LET. V., THE COUNTESS OF B——., Nuremberg, Aug. 22. O. S. 1716., textoftheletter 
"""
import csv

# Define the folder where the letters are stored
letters_folder = "../../output/montagu_letters" 
output_file = "../../output/montagu_letters_unedited.csv"

# Helper function to extract the numeric index from the filename
def get_letter_number(filename):
    # Matches filenames like "letter_1.txt" -> returns integer 1
    match = re.match(r"letter_(\d+)\.txt", filename)
    if match:
        return int(match.group(1))
    return float("inf")  # Fallback if it doesn't match the pattern

# List all text files in the folder, then sort numerically
letter_files = [f for f in os.listdir(letters_folder) if f.startswith("letter_") and f.endswith(".txt")]
letter_files.sort(key=get_letter_number)

# Prepare the CSV headers
headers = ["filename", "title", "addressee", "location", "date", "body"]

# Initialize storage for letters
letters_data = []

# Loop through each letter file
for letter_file in letter_files:
    file_path = os.path.join(letters_folder, letter_file)
    print(f"Processing: {file_path}")

    # Read the content of the letter
    with open(file_path, "r", encoding="utf-8") as file:
        lines = [line.strip() for line in file.readlines() if line.strip()]

    if not lines:
        continue  # Skip empty files

    # Extract filename
    filename = letter_file

    # Extract title (first line)
    title = lines[0]

    # Extract addressee (line starting with "To ", "to ", or "TO ")
    addressee = "Unknown"
    addressee_index = None
    for idx, line in enumerate(lines):
        if re.match(r"^(To|TO|to)\s+(.*)", line):
            addressee = line.strip()
            addressee_index = idx
            break

    # Extract location and date (first line after addressee, ignoring footnotes)
    location, date = "Unknown", "Unknown"
    location_date_line = None

    if addressee_index is not None:
        for line in lines[addressee_index + 1:]:  # start after the addressee line
            if "Footnote:" in line:
                continue  # skip footnotes
            location_date_line = line
            break

    if location_date_line:
        # Split by the first comma
        parts = location_date_line.split(",", 1)
        if len(parts) == 2:
            location = parts[0].strip()
            date = parts[1].strip()

    # Extract body (everything after location/date line)
    if location_date_line and location_date_line in lines:
        body_start_idx = lines.index(location_date_line) + 1
    else:
        body_start_idx = addressee_index + 1 if addressee_index is not None else 1

    body = "\n".join(lines[body_start_idx:]).strip()

    # Store data
    letters_data.append([filename, title, addressee, location, date, body])

# Write data to CSV file
with open(output_file, "w", encoding="utf-8", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(headers)  # Write header row
    writer.writerows(letters_data)  # Write letter rows

print(f"CSV file '{output_file}' has been created successfully.")


After running this code, we ended up with 51 letters correctly extracted and only 7 needing manual editing. 

In [ ]:
import pandas as pd

# Load the CSV file into a DataFrame
montagu_unedited = pd.read_csv('../../output/montagu_letters_unedited.csv')
montagu_unedited.head()

In [ ]:
# Let's make sure that we got all the 58 letters
montagu_unedited.shape

In [ ]:
# Some descriptive statistics, especially relevant are unique counts of addressee, location, and date
montagu_unedited.describe()

In [ ]:
# What are the data types of the columns?
print(montagu_unedited.dtypes)

In [ ]:
# Check for missing values in the DataFrame
print(montagu_unedited.isnull().sum())

In [ ]:
# We could have also done this with the `info()` method
montagu_unedited.info()

In [ ]:
# This seems fine?? Are we ready to move on?

# Let's see if we have any unkowns
print('addressee')
print(montagu_unedited[montagu_unedited['addressee'] == 'Unknown'])

print('location')
print(montagu_unedited[montagu_unedited['location'] == 'Unknown'])

print('date')
print(montagu_unedited[montagu_unedited['date'] == 'Unknown'])

<div style="border-left: 4px solid #60a5fa; padding: 0.5em 1em; margin: 1.5em 0;">
<strong>Note on the 'Unknown'</strong>

<p>We employed this trick in our code, where we first created everything with the string <code>'Unknown'</code> and then filled it in with the correct information if the information was there. Here is the snippet:</p>

<pre><code>addressee = "Unknown"
addressee_index = None
for idx, line in enumerate(lines):
    if re.match(r"^(To|TO|to)\s+(.*)", line):
        addressee = line.strip()
        addressee_index = idx
        break</code></pre>

<p>The main reason for this decision was human readability. We knew that the next steps would include data editing and augmentation, so we wanted to make sure that we knew these columns were not intentionally left empty (meaning there is no information such as no date or no location).</p>
</div>


## Editing

While it is possible to tackle this issue of the unknowns with code, we also shouldn't forget that **manual data cleaning** is always an option. Besides we will learn a lot about our dataset by studying it manually.

In this section, we moved our unedited csv to `Google Sheets` for some minor edits. You can do this in any csv editor of your choice, including in VS Code. We had an easier time editing this collaboratively in an online platform.

List of edits:

1. Rows where the text is not split accurately
2. Unknowns
3. Dates and References
4. Cross-referencing letters with other editions

In [ ]:
# This is the cleaned and not yet augmented version of our data, which we called v1

montagu_v1 = pd.read_csv('../../output/montagu_letters_v1.csv')
montagu_v1.head()

### Edit 1: Issues with Splitting

In [ ]:
# Helper function to Highlight rows where values differ
def highlight_diff(row):
    color = 'background: rgba(167, 139, 250, 0.2);' if row['Before'] != row['After'] else ''
    return [color, color]

In [ ]:
# Get letter 41 from both DataFrames
row_before = montagu_unedited[montagu_unedited['filename'] == 'letter_41.txt'].iloc[0]
row_after = montagu_v1[montagu_v1['filename'] == 'letter_41.txt'].iloc[0]


In [ ]:
from IPython.display import HTML

# Union of both column orders, preserving order
fields = list(dict.fromkeys(list(row_before.index) + list(row_after.index)))

rows = ""
for f in fields:
    b = str(row_before[f]) if f in row_before.index else "—"
    a = str(row_after[f]) if f in row_after.index else "—"
    b_display = b[:300] + '…' if f == 'body' and len(b) > 300 else b
    a_display = a[:300] + '…' if f == 'body' and len(a) > 300 else a
    bg = "background: rgba(167, 139, 250, 0.2);" if b != a else ""
    rows += f"""
    <tr style="{bg}">
        <td style="padding: 8px 14px; font-weight: 600; opacity: 0.7; white-space: nowrap;">{f}</td>
        <td style="padding: 8px 14px; max-width: 400px; word-wrap: break-word;">{b_display}</td>
        <td style="padding: 8px 14px; max-width: 400px; word-wrap: break-word;">{a_display}</td>
    </tr>"""

HTML(f"""
<table style="border-collapse: collapse; width: 100%; font-size: 0.95em;">
  <thead>
    <tr style="border-bottom: 2px solid #a78bfa;">
      <th style="padding: 8px 14px; text-align: left;">Field</th>
      <th style="padding: 8px 14px; text-align: left;">Before</th>
      <th style="padding: 8px 14px; text-align: left;">After</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
""")


### Edit 2: Unknowns

In [ ]:
# Get letter 53 from both DataFrames
# Note that we are using the same variable, so it overrides what we did earlier. 
# This is ok because we use this code for demonstration purposes only

row_before = montagu_unedited[montagu_unedited['filename'] == 'letter_53.txt'].iloc[0]
row_after = montagu_v1[montagu_v1['filename'] == 'letter_53.txt'].iloc[0]

In [ ]:
# Union of both column orders, preserving order
fields = list(dict.fromkeys(list(row_before.index) + list(row_after.index)))

rows = ""
for f in fields:
    b = str(row_before[f]) if f in row_before.index else "—"
    a = str(row_after[f]) if f in row_after.index else "—"
    b_display = b[:300] + '…' if f == 'body' and len(b) > 300 else b
    a_display = a[:300] + '…' if f == 'body' and len(a) > 300 else a
    bg = "background: rgba(167, 139, 250, 0.2);" if b != a else ""
    rows += f"""
    <tr style="{bg}">
        <td style="padding: 8px 14px; font-weight: 600; opacity: 0.7; white-space: nowrap;">{f}</td>
        <td style="padding: 8px 14px; max-width: 400px; word-wrap: break-word;">{b_display}</td>
        <td style="padding: 8px 14px; max-width: 400px; word-wrap: break-word;">{a_display}</td>
    </tr>"""

HTML(f"""
<table style="border-collapse: collapse; width: 100%; font-size: 0.95em;">
  <thead>
    <tr style="border-bottom: 2px solid #a78bfa;">
      <th style="padding: 8px 14px; text-align: left;">Field</th>
      <th style="padding: 8px 14px; text-align: left;">Before</th>
      <th style="padding: 8px 14px; text-align: left;">After</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
""")


### Edit 3: Dates & References

As we see in the examples above, we removed the original `date` column and instead created two columns: `date_original` which retains the information from the `date` column and `date_edited`. In this second column, we corrected some minor errors in dates following the Broadview edition (see Readings/Bibliography) and standardized the dating format to: **YYYY-MM-DD**

Similarly, we created a new column called `references`. For each letter, we identified page numbers from the Broadview edition. This helped us connect our research with the newest and most reputable scholarly edition of these letters

### Edit 4:

This was a difficult one! There are 6 additional letters (53 through 58) in the 1790 edition which were not present in the 1763 version. Lord Wharncliffe, who edited the entire correspondence and works of Lady Montagu in 1861 pointed out that all but the Letter 58 are not real. Letter 58 was published in newspapers in 1719. Considering the fact that the Broadview edition and the 1764 French translation of this work do not include these 6 letters, we decided to remove them from the corpus. We are however keeping the text files in our `output/montagu_letters` folder for further analysis.

In [ ]:
montagu_v2 = montagu_v1.iloc[:-6]
montagu_v2.to_csv('../../output/montagu_letters_v2.csv', index=False)
print(montagu_v2.shape)


We have now 52 letters, cleaned up and ready for augmentation!

## Data Enrichment

Data enrichment is the process through which an existing dataset is enhanced by adding supplementary information to make records more complete and useful for analysis.

<div style="background: rgba(167, 139, 250, 0.12); border-radius: 6px; padding: 0.6em 1em; margin: 1.5em auto; font-weight: 500; width: 70%;">
  Data enrichment at this stage of the corpus focuses on the metadata, specifically on locations and people.
</div>

### Locations

We have the locations that the letters were sent from! 
    
These are historical places. Not all of these locations have the same names in the 21st century as they did in the 18th, like *"Brunswick"* (Letter 16). 

The data enrichment process for locations includes finding the current names of these places and adding **Wikidata QIDs** as well as coordinates from Wikidata. Wikidata QIDs are unique identifiers that help us connect our data to that of others. For example, another dataset containing letters from other travellers could be linked to our dataset through the QIDs of the places that they travelled to.

### People

We have the addressees that the letters were sent to!

The Gutenberg volume that we extracted the texts from is identical to the original 1790 edition. This means that we have the names of the addressees in this 18th century, partly anonymized style, like: "TO THE COUNTESS OF B----." (Letter 5)

However, in more contemporary editions of Montagu's letters, different scholars have identified who the recipients were. We crossference these editions to update our addressee column.

Additionally, we will use *Early Modern Letters Online* too collect unique identifiers for the people in this dataset. Some of the letter recipients are not well-known enough to have their own Wikidata pages and EMLO is reliable and  well-suited for our corpus of Early Modern letters.

### How to approach data enrichment?

There are many ways to approach the enrichment process, including automated methods that use Wikidata API for queries. However, we chose to approach this problem manually and use Google Sheets. Since we have only a handful letters and a lot of the addressees and locations are repeated, it was faster and more reliable to do it ourselves. 

> Later in this course, we will see more automated approaches to identifying people and place names in the bodies of the letters.